# Elastic Net Regression - Starter Notebook
Elastic Net combines L1 (Lasso) and L2 (Ridge) penalties, balancing feature selection and coefficient stability.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import ElasticNet, ElasticNetCV, Lasso, Ridge, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.datasets import make_regression

In [ ]:
np.random.seed(42)
# Highly correlated features + sparse true model
X, y, coef_true = make_regression(n_samples=300, n_features=30, n_informative=8,
                                   noise=15, coef=True, random_state=42)
# Add correlated duplicates
X = np.hstack([X, X[:, :10] + np.random.randn(300, 10) * 0.1])

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)
print('Feature shape:', X_tr_s.shape)

In [ ]:
# Compare OLS / Ridge / Lasso / Elastic Net
models = {
    'OLS':          LinearRegression(),
    'Ridge(α=1)':   Ridge(alpha=1.0),
    'Lasso(α=0.5)': Lasso(alpha=0.5, max_iter=10000),
    'EN(α=0.5,l1=0.5)': ElasticNet(alpha=0.5, l1_ratio=0.5, max_iter=10000),
    'EN(α=0.5,l1=0.1)': ElasticNet(alpha=0.5, l1_ratio=0.1, max_iter=10000),
    'EN(α=0.5,l1=0.9)': ElasticNet(alpha=0.5, l1_ratio=0.9, max_iter=10000),
}
rows = []
for name, m in models.items():
    m.fit(X_tr_s, y_tr)
    mse = mean_squared_error(y_te, m.predict(X_te_s))
    nz = np.sum(m.coef_ != 0)
    rows.append({'Model': name, 'Test MSE': round(mse, 2), 'Non-zero coefs': nz})

print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Elastic Net CV to find best alpha and l1_ratio
en_cv = ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
                     alphas=np.logspace(-3, 1, 50),
                     cv=5, max_iter=10000, random_state=42)
en_cv.fit(X_tr_s, y_tr)
print(f'\nBest alpha:    {en_cv.alpha_:.4f}')
print(f'Best l1_ratio: {en_cv.l1_ratio_:.2f}')
print(f'Non-zero coefs: {np.sum(en_cv.coef_ != 0)}')
print(f'Test MSE: {mean_squared_error(y_te, en_cv.predict(X_te_s)):.2f}')

In [ ]:
# Coefficient comparison plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
models_plot = {
    'Lasso':        Lasso(alpha=0.5, max_iter=10000).fit(X_tr_s, y_tr),
    'Ridge':        Ridge(alpha=1.0).fit(X_tr_s, y_tr),
    'Elastic Net':  en_cv,
}
for ax, (name, m) in zip(axes, models_plot.items()):
    ax.bar(range(len(m.coef_)), m.coef_, color='steelblue', edgecolor='none')
    ax.set_title(f'{name}\nnon-zero={np.sum(m.coef_!=0)}')
    ax.set_xlabel('Feature index')
    ax.set_ylabel('Coefficient')
    ax.axhline(0, color='black', linewidth=0.7)

plt.suptitle('Coefficient Profiles: Lasso vs Ridge vs Elastic Net', fontsize=11)
plt.tight_layout()
plt.show()